In [27]:
import json

# Load the result dictionary from the JSON file
with open('/Users/kesiyun/Desktop/00workspace/00sequenceresult.json', 'r') as f:
    loaded_result = json.load(f)

#for key, value in loaded_result.items():
    #print(key,': ',value)


In [28]:
#load utilities
import os
import re
import json
import subprocess
from typing import List, Dict, Tuple, Optional
import torch
import json
import os
import cv2
import numpy as np
import math
import re

from datetime import datetime, timedelta

def get_video_duration_seconds(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    #print(fps)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    cap.release()
    return duration

def timestamp_to_clip_index(timestamp, length_of_clips):
    # Parse the timestamp string (e.g., "0:03")
    minutes, seconds = map(int, timestamp.split(':'))
    total_seconds = minutes * 60 + seconds

    # Calculate which clip this timestamp falls into
    clip_index = total_seconds // length_of_clips

    return clip_index

# Example usage:
#print(timestamp_to_clip_index("0:03", 4))  # ➜ 0
#print(timestamp_to_clip_index("0:05", 3))  # ➜ 1
#print(timestamp_to_clip_index("0:10", 5))  # ➜ 2


def get_timestamp(length_of_clip, clip_index):
    total_seconds = length_of_clip * clip_index
    minutes = total_seconds // 60
    seconds = total_seconds % 60
    return f"{minutes}:{seconds:02d}"


# Examples:
#print(get_timestamp(5, 0))  # Output: 0:00
#print(get_timestamp(5, 1))  # Output: 0:05
#print(get_timestamp(5, 3))  # Output: 0:15



def find_contractions(list_of_clips):
    C_locations=[]
    for i in range(len(list_of_clips)):
        if 'C' in list_of_clips[i]:
            C_locations.append(i)
    return C_locations



def timestamp_to_seconds_temp(timestamp): #unused but reserved
    minutes, seconds = map(int, timestamp.split(':'))
    return minutes * 60 + seconds

def timestamp_to_seconds(ts):
    """Convert 'M:SS' or 'H:MM:SS' to seconds."""
    parts = list(map(int, ts.split(':')))
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    elif len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    else:
        raise ValueError("Invalid timestamp format")


def extract_pairs(input_string):
    # Example usage
    #input_str = "R P(0:00) A(0:01)C(0:02) P(1:47) A(1:47)C(1:48)"
    #result = extract_pairs(input_str) #[['R', '0:00'], ['P', '0:00'], ['A', '0:01'], ['C', '0:02'], ['P', '1:47'], ['A', '1:47'], ['C', '1:48']]
    # Always start with 'R' at '0:00'
    pairs = [['R', '0:00']]
    
    # Find all matches like A(1:38), C(1:52), etc.
    matches = re.findall(r'([A-Z])\((\d+:\d+)\)', input_string)
    
    # Append each match as a [letter, timestamp] pair
    for letter, timestamp in matches:
        pairs.append([letter, timestamp])
    
    return pairs



def is_timestamp_covered(reference_list, target_timestamp):
    #reference = ['0:00', '2:47', '4:48']
    #print(is_timestamp_covered(reference, '2:48'))  # False
    #print(is_timestamp_covered(reference, '2:43'))  # True
    #print(is_timestamp_covered(reference, '2:42'))  # True
    #print(is_timestamp_covered(reference, '2:41'))  # False
    # Convert target timestamp to datetime object
    target_minutes, target_seconds = map(int, target_timestamp.split(":"))
    target_total_seconds = target_minutes * 60 + target_seconds
    end_total_seconds = target_total_seconds + 5

    for ref in reference_list:
        ref_minutes, ref_seconds = map(int, ref.split(":"))
        ref_total_seconds = ref_minutes * 60 + ref_seconds
        if target_total_seconds <= ref_total_seconds <= end_total_seconds:
            return True
    return False

import time
import sys

def simple_progress_bar(current, total=1):
    if current=='writing':
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        sys.stdout.write(f'\rwriting')
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        return
    if current=='saved':
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        sys.stdout.write(f'\rsaved  ')
        sys.stdout.flush() # 刷新输出缓冲区，确保立即显示
        return
        
    rate = float(current) / total
    arrow = '-' * int(rate * 50) + '>'  # 进度条的视觉表示
    percentage = int(rate * 100)
    # 使用 \r 将光标移到行首，覆盖之前的输出
    sys.stdout.write(f'\r[{arrow:<50}] {percentage}%')
    sys.stdout.flush() # 刷新输出缓冲区，确保立即显示






In [29]:
# specify location of profiles
#REMARK: ONLY PROCESSED UNTIL LAST CLIP (i.e. LAST NOTATIONS IN mmc2.xlsx)

virtual_clip_dict_profile='/Users/kesiyun/Desktop/00workspace/00full_clips_5s.json'#'/content/drive/MyDrive/Colab Notebooks/sten/00virtual_clips_5s.json'
video_split_profile='/Users/kesiyun/Desktop/00workspace/00full_video_split_5s.json'#'/content/drive/MyDrive/Colab Notebooks/sten/00video_split_5s.json'
sequence_result_profile='/Users/kesiyun/Desktop/00workspace/00sequenceresult.json'#'/content/drive/MyDrive/Colab Notebooks/sten/00sequenceresult.json'

In [30]:
#load video and profiles
import json
with open(sequence_result_profile, 'r') as f:
    loaded_sequence = json.load(f)
with open(virtual_clip_dict_profile, 'r') as f:
    loaded_virtual_clip_dict = json.load(f)
with open(video_split_profile, 'r') as f:
    loaded_split_data = json.load(f)

testing_set = loaded_split_data['testing_set']
training_set = loaded_split_data['training_set']

# count Training-Testing configurations
##total
total_clips=0
key_count=0
for key, value in loaded_virtual_clip_dict.items():
    #print(key,": ",value['num_of_clips'] )
    total_clips+=value['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each)\n')

##Testing
print("Testing Set:", testing_set)
total_clips=0
key_count=0
for videos in testing_set:
    total_clips+=loaded_virtual_clip_dict[videos]['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each) \n')

##Training
print("Training Set:", training_set)
total_clips=0
key_count=0
for videos in training_set:
    total_clips+=loaded_virtual_clip_dict[videos]['num_of_clips']
    key_count+=1
print(total_clips, 'clips in',key_count,'videos (average',total_clips/key_count,'each) \n')



8881 clips in 57 videos (average 155.80701754385964 each)

Testing Set: ['16A', '4C', '4A', '4E', '12B', '17I']
1644 clips in 6 videos (average 274.0 each) 

Training Set: ['1A', '1B', '1C', '1D', '1E', '1F', '2A', '3A', '3B', '3C', '3D', '4B', '4D', '4F', '5A', '6A', '6B', '6C', '6D', '7A', '8A', '8B', '9A', '10A', '10B', '10C', '11A', '11B', '11C', '11D', '12A', '13A', '14A', '14B', '14C', '14D', '14E', '15A', '15B', '15C', '15D', '17A', '17B', '17C', '17D', '17E', '17F', '17G', '17H', '18A', '18B']
7237 clips in 51 videos (average 141.90196078431373 each) 



In [31]:
#set video here except for batch
length_of_clip=5
videos_dir='/Volumes/Green SSD/00Workspace_portable/videos/' ####
#videos_dir='/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/temp_source'
videos_dir='/Volumes/Green SSD/00Workspace_portable/segmented_videos/whiteout'
video_name='1A'#training_set[2]#'1A' ####

#avi_path=videos_dir+'/'+video_name+'.avi'
avi_path = os.path.join(videos_dir, video_name + '.avi')
avi_path = os.path.join(videos_dir, video_name + '.mp4')


#print referrence
print('Video file:',avi_path)
print('Sequence:', loaded_sequence[video_name])
#print('Clips (',length_of_clip,'s):', loaded_virtual_clip_dict[video_name],)
#list_of_contractions=find_contractions(loaded_virtual_clip_dict[video_name]['list_of_clips'])
#print('Contractions in:',list_of_contractions)

Video file: /Volumes/Green SSD/00Workspace_portable/segmented_videos/whiteout/1A.mp4
Sequence: {'abstract': 'R P ACD', 'timesec': 'R P(0:39) A(1:38)C(1:52)D(4:25)'}


In [34]:

import os
import re
import json
import subprocess
from typing import List, Dict, Tuple, Optional


# --- Your training set (example) ---
#training_set = ["18A", "18B"]  # e.g., ["1A", "1B", "1C", "18A", "18B"]

# --- Output settings ---
output_ext = ".avi"   # Keep .avi as requested (set ".mp4" if you want H.264)
precise = True        # True: exact boundaries via re-encode; False: fast stream copy (keyframe-limited)

# Ensure output folder exists
#output_dir= '/Volumes/Green SSD/00Workspace_portable/videos_aug/'
#output_dir= '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug'
output_dir= '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj'

from datetime import datetime, timezone
import tempfile
import shutil

# Tracking file (in output directory)
processed_index_filename = "processed.json"
processed_index_path = os.path.join(output_dir, processed_index_filename)

# Control whether to skip previously processed videos
skip_processed = True

# Optional: force reprocessing even if processed.json says done
force_reprocess = False  # set True when you want to redo everything



In [35]:

def load_sequence_map(path: str) -> Dict[str, Dict[str, str]]:
    """Load the dict mapping video_name -> {'abstract': ..., 'timesec': ...}."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Sequence profile not found: {path}")
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def parse_c_timestamps(timesec: str) -> List[int]:
    """
    Extract seconds for all C(mm:ss) occurrences from a 'timesec' string.
    Works with concatenated entries like '...C(7:50)C(9:47)...'.
    """
    if not timesec or not isinstance(timesec, str):
        return []
    pattern = r'C\((\d+):([0-5]\d)\)'
    matches = re.findall(pattern, timesec)
    seconds_list = []
    for m_str, s_str in matches:
        try:
            total = int(m_str) * 60 + int(s_str)
            seconds_list.append(total)
        except ValueError:
            pass
    # Deduplicate while preserving order
    seen, uniq = set(), []
    for t in seconds_list:
        if t not in seen:
            uniq.append(t)
            seen.add(t)
    return uniq

def ffprobe_duration_seconds(input_path: str) -> Optional[float]:
    """
    Use ffprobe to get duration in seconds.
    Returns None if ffprobe is not available or fails.
    """
    try:
        cmd = [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            input_path
        ]
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT)
        dur = float(out.strip())
        return dur
    except Exception:
        return None

def format_mmss(ts_seconds: int) -> str:
    m = ts_seconds // 60
    s = ts_seconds % 60
    return f"{m:02d}{s:02d}"

def clip_window_around_c(c_seconds: int, duration: float, half_window: int = 5) -> Tuple[float, float]:
    """
    Compute [start, end] around the C time, clamped to [0, duration].
      - Normally [C-5, C+5]
      - Near start or end, clamp appropriately
    """
    start = max(0.0, c_seconds - half_window)
    end = min(duration, c_seconds + half_window)
    if end <= start:
        end = min(duration, start + half_window * 2)
    return float(start), float(end)

def cut_clip_ffmpeg(input_path: str, start_s: float, end_s: float, output_path: str,
                    precise: bool, output_ext: str) -> Tuple[bool, str]:
    """
    Cut [start_s, end_s] from input_path and write to output_path with ffmpeg.
    Returns (success, message).
    - precise=True: re-encode for exact boundaries.
    - precise=False: stream copy (fast, keyframe-limited).
    """
    length = max(0.0, end_s - start_s)
    if length <= 0.0:
        return False, "Clip length <= 0"

    args = ["ffmpeg", "-y", "-ss", f"{start_s:.3f}", "-i", input_path, "-t", f"{length:.3f}"]

    if precise:
        if output_ext.lower() == ".avi":
            # Use codecs compatible with AVI container
            args += [
                "-c:v", "mpeg4",
                "-qscale:v", "4",        # quality factor (2–5 are good)
                "-c:a", "mp3",           # audio codec compatible with AVI
                "-b:a", "192k",
                "-movflags", "+faststart",
                "-avoid_negative_ts", "make_zero"
            ]
        else:
            # Defaults for MP4/H.264
            args += [
                "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
                "-c:a", "aac", "-b:a", "128k",
                "-pix_fmt", "yuv420p",
                "-movflags", "+faststart",
                "-avoid_negative_ts", "make_zero"
            ]
    else:
        args += ["-c", "copy"]

    args += [output_path]

    try:
        subprocess.run(args, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True, f"Wrote {output_path}"
    except subprocess.CalledProcessError as e:
        msg = e.stderr.decode('utf-8', errors='ignore')
        return False, f"ffmpeg error: {msg[:500]}"


def _now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def load_processed_index(path: str) -> Dict:
    """
    Load processed.json. If missing or invalid, return a fresh structure.
    """
    if not os.path.exists(path):
        return {
            "processed_videos": [],
            "meta": {"last_updated": _now_iso(), "version": 1}
        }
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        # basic validation
        if "processed_videos" not in data or not isinstance(data["processed_videos"], list):
            raise ValueError("Invalid processed.json structure")
        return data
    except Exception:
        # fall back to fresh index if file is corrupted
        return {
            "processed_videos": [],
            "meta": {"last_updated": _now_iso(), "version": 1}
        }

def save_processed_index(path: str, data: Dict) -> None:
    """
    Atomically write processed.json to disk (write to temp then rename).
    """
    data["meta"]["last_updated"] = _now_iso()
    dir_path = os.path.dirname(path)
    os.makedirs(dir_path, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=dir_path, prefix="processed_", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        shutil.move(tmp, path)
    finally:
        # If tmp still exists due to an exception, try cleaning it up
        try:
            if os.path.exists(tmp):
                os.remove(tmp)
        except Exception:
            pass

def mark_video_processed(index: Dict, video_name: str) -> None:
    """
    Add video_name to processed_videos (if not already present).
    """
    pv = index.setdefault("processed_videos", [])
    if video_name not in pv:
        pv.append(video_name)

def is_video_processed(index: Dict, video_name: str) -> bool:
    """
    Check whether video_name is in processed.json.
    """
    pv = index.get("processed_videos", [])
    return video_name in pv



In [36]:

def process_videos(training_set: List[str],
                   videos_dir: str,
                   output_dir: str,
                   seq_map: Dict[str, Dict[str, str]],
                   output_ext: str = ".avi",
                   precise: bool = True,
                   processed_index_path: str = processed_index_path,
                   skip_processed: bool = True,
                   force_reprocess: bool = False,
                   mark_only_on_success: bool = False) -> None:
    """
    Iterate through training_set, parse C() timestamps, clip windows, and write outputs.
    Adds processed tracking via processed.json in output_dir:
      - skip_processed: if True, skip videos already in processed.json
      - force_reprocess: if True, ignore processed.json and redo all
      - mark_only_on_success: if True, mark processed only if all clips succeed,
        otherwise mark processed after attempting (default False).
    """
    # Load processed state
    index = load_processed_index(processed_index_path)
    print(f"[INDEX] Loaded processed index with {len(index.get('processed_videos', []))} videos.")

    for video_name in training_set:
        # Skip if already processed
        if skip_processed and not force_reprocess and is_video_processed(index, video_name):
            print(f"[SKIP] {video_name} is already marked processed.")
            continue

        input_path = os.path.join(videos_dir, f"{video_name}.mp4")#.avi")
        if not os.path.exists(input_path):
            print(f"[WARN] Missing AVI: {input_path}")
            continue

        meta = seq_map.get(video_name)
        if not meta or "timesec" not in meta:
            print(f"[WARN] No timesec for {video_name}")
            continue

        timesec_str = meta["timesec"]
        c_times = parse_c_timestamps(timesec_str)
        if not c_times:
            print(f"[INFO] No C() timestamps in {video_name} — nothing to clip.")
            # Decision: still mark processed, since there's nothing to do
            mark_video_processed(index, video_name)
            save_processed_index(processed_index_path, index)
            print(f"[INDEX] Marked processed: {video_name} (no clips).")
            continue

        duration = ffprobe_duration_seconds(input_path)
        if duration is None:
            print(f"[WARN] Could not read duration via ffprobe for {input_path}. Is ffmpeg installed and in PATH?")
            continue

        print(f"\n[PROCESS] {video_name}: duration={duration:.2f}s | C count={len(c_times)}")

        # Track whether all clips succeed (only used if mark_only_on_success=True)
        all_ok = True

        for c_sec in c_times:
            start_s, end_s = clip_window_around_c(c_sec, duration, half_window=5)
            tag = format_mmss(c_sec)
            out_name = f"{video_name}_{tag}{output_ext}"
            output_path = os.path.join(output_dir, out_name)

            ok, msg = cut_clip_ffmpeg(
                input_path=input_path,
                start_s=start_s,
                end_s=end_s,
                output_path=output_path,
                precise=precise,
                output_ext=output_ext
            )
            if ok:
                print(f"  [OK] C at {tag} → [{start_s:.2f}, {end_s:.2f}] → {out_name}")
            else:
                all_ok = False
                print(f"  [FAIL] C at {tag} → {msg}")

        # Mark processed depending on policy
        if mark_only_on_success:
            if all_ok:
                mark_video_processed(index, video_name)
                save_processed_index(processed_index_path, index)
                print(f"[INDEX] Marked processed: {video_name} (all clips OK).")
            else:
                print(f"[INDEX] NOT marking {video_name} (some clips failed).")
        else:
            # Mark as processed after attempting (even with failures)
            mark_video_processed(index, video_name)
            save_processed_index(processed_index_path, index)
            print(f"[INDEX] Marked processed: {video_name} (attempted).")


In [37]:

# If you already have `loaded_sequence` in the notebook, use it:
# seq_map = loaded_sequence

# Otherwise, load from JSON file:
seq_map = load_sequence_map(sequence_result_profile)

process_videos(
    training_set=training_set,
    videos_dir=videos_dir,
    output_dir=output_dir,
    seq_map=seq_map,
    output_ext=output_ext,
    precise=precise,
    processed_index_path=processed_index_path,
    skip_processed=skip_processed,
    force_reprocess=force_reprocess,      # set True to rerun everything
    mark_only_on_success=False             # set True if you only want to mark on full success
)


[INDEX] Loaded processed index with 0 videos.

[PROCESS] 1A: duration=267.00s | C count=1
  [OK] C at 0152 → [107.00, 117.00] → 1A_0152.avi
[INDEX] Marked processed: 1A (attempted).

[PROCESS] 1B: duration=313.71s | C count=1
  [OK] C at 0325 → [200.00, 210.00] → 1B_0325.avi
[INDEX] Marked processed: 1B (attempted).

[PROCESS] 1C: duration=720.57s | C count=4
  [OK] C at 0036 → [31.00, 41.00] → 1C_0036.avi
  [OK] C at 0150 → [105.00, 115.00] → 1C_0150.avi
  [OK] C at 0550 → [345.00, 355.00] → 1C_0550.avi
  [OK] C at 0930 → [565.00, 575.00] → 1C_0930.avi
[INDEX] Marked processed: 1C (attempted).
[WARN] Missing AVI: /Volumes/Green SSD/00Workspace_portable/segmented_videos/whiteout/1D.mp4

[PROCESS] 1E: duration=272.43s | C count=2
  [OK] C at 0040 → [35.00, 45.00] → 1E_0040.avi
  [OK] C at 0247 → [162.00, 172.00] → 1E_0247.avi
[INDEX] Marked processed: 1E (attempted).

[PROCESS] 1F: duration=264.57s | C count=1
  [OK] C at 0121 → [76.00, 86.00] → 1F_0121.avi
[INDEX] Marked processed: 1F 

In [38]:
#rotate


import os
import subprocess
from typing import List, Tuple

# Your output folder from the previous step:
#output_dir = '/Volumes/Green SSD/00Workspace_portable/videos_aug/'
#output_dir= '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug'
output_dir= '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj'


# Only process files with this extension:
ext = ".avi"

# ffmpeg re-encode settings compatible with AVI
VIDEO_CODEC = "mpeg4"
VIDEO_QUALITY = "4"      # qscale (2-5 are good; lower = better quality, larger files)
AUDIO_CODEC = "mp3"
AUDIO_BITRATE = "192k"

# Dry-run: set True to only print actions without writing files
dry_run = False


In [39]:

def is_avi(path: str) -> bool:
    return path.lower().endswith(ext)

def split_name(path: str) -> Tuple[str, str]:
    """
    Return (basename_without_ext, extension) for a full path.
    """
    base = os.path.basename(path)
    root, extn = os.path.splitext(base)
    return root, extn

def has_suffix(root: str) -> bool:
    """
    Check if the base name already ends with _A/_B/_C/_D
    """
    return root.endswith(("_A", "_B", "_C", "_D"))

def run_ffmpeg(args: List[str]) -> Tuple[bool, str]:
    """
    Execute ffmpeg command. Returns (ok, message).
    """
    if dry_run:
        return True, "[DRY-RUN] " + " ".join(args)
    try:
        subprocess.run(args, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True, "OK"
    except subprocess.CalledProcessError as e:
        msg = e.stderr.decode("utf-8", errors="ignore")
        return False, msg[:500]

def make_rotated(input_path: str, output_path: str, vf_expr: str) -> Tuple[bool, str]:
    """
    Create rotated video using given vf (video filter) expression.
    """
    args = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-vf", vf_expr,
        "-c:v", VIDEO_CODEC, "-qscale:v", VIDEO_QUALITY,
        "-c:a", AUDIO_CODEC, "-b:a", AUDIO_BITRATE,
        "-movflags", "+faststart",
        "-avoid_negative_ts", "make_zero",
        output_path
    ]
    return run_ffmpeg(args)


In [40]:

def process_rotations(output_dir: str):
    """
    For each AVI in output_dir:
      - If filename has no _[A-D] suffix, rename to _A.
      - From the _A file, create rotated versions:
          _B (90° CW), _C (180°), _D (270° CW)
      - Skip already existing outputs.
    """
    files = [f for f in os.listdir(output_dir) if is_avi(f)]
    files.sort()

    if not files:
        print("[INFO] No AVI files found.")
        return

    print(f"[INFO] Found {len(files)} AVI files in {output_dir}")

    for f in files:
        full_path = os.path.join(output_dir, f)
        root, extn = split_name(full_path)  # extn like '.avi'

        # Determine base name and current suffix status
        if has_suffix(root):
            base_no_suffix = root[:-2]  # strip _A/_B/_C/_D
            current_suffix = root[-1]   # 'A'/'B'/'C'/'D'
        else:
            base_no_suffix = root
            current_suffix = None

        # 1) If untouched (no suffix), rename to _A
        if current_suffix is None:
            new_root = base_no_suffix + "_A"
            new_name = new_root + extn
            new_path = os.path.join(output_dir, new_name)

            print(f"[RENAME] {f} → {new_name}")
            if not dry_run:
                os.rename(full_path, new_path)

            # Now set variables to continue rotation from the newly renamed file
            root = new_root
            full_path = new_path
            current_suffix = "A"

        # 2) Build target names for rotations relative to the _A file
        a_path = os.path.join(output_dir, base_no_suffix + "_A" + extn)
        b_path = os.path.join(output_dir, base_no_suffix + "_B" + extn)
        c_path = os.path.join(output_dir, base_no_suffix + "_C" + extn)
        d_path = os.path.join(output_dir, base_no_suffix + "_D" + extn)

        # If _A doesn't exist (e.g., only B/C/D present somehow), skip safely
        if not os.path.exists(a_path):
            print(f"[WARN] Missing _A source for {base_no_suffix}, skipping rotations.")
            continue

        # 3) Create _B (90° clockwise)
        if not os.path.exists(b_path):
            print(f"[ROTATE] _B (90° CW): {os.path.basename(a_path)} → {os.path.basename(b_path)}")
            ok, msg = make_rotated(a_path, b_path, vf_expr="transpose=1")
            print("   ", "OK" if ok else f"FAIL: {msg}")
        else:
            print(f"[SKIP] Exists: {os.path.basename(b_path)}")

        # 4) Create _C (180°)
        if not os.path.exists(c_path):
            print(f"[ROTATE] _C (180°): {os.path.basename(a_path)} → {os.path.basename(c_path)}")
            ok, msg = make_rotated(a_path, c_path, vf_expr="hflip,vflip")
            print("   ", "OK" if ok else f"FAIL: {msg}")
        else:
            print(f"[SKIP] Exists: {os.path.basename(c_path)}")

        # 5) Create _D (270° CW i.e. 90° CCW)
        if not os.path.exists(d_path):
            print(f"[ROTATE] _D (270° CW): {os.path.basename(a_path)} → {os.path.basename(d_path)}")
            ok, msg = make_rotated(a_path, d_path, vf_expr="transpose=2")
            print("   ", "OK" if ok else f"FAIL: {msg}")
        else:
            print(f"[SKIP] Exists: {os.path.basename(d_path)}")


In [41]:
process_rotations(output_dir)

[INFO] Found 77 AVI files in /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj
[RENAME] 10A_0016.avi → 10A_0016_A.avi
[ROTATE] _B (90° CW): 10A_0016_A.avi → 10A_0016_B.avi
    OK
[ROTATE] _C (180°): 10A_0016_A.avi → 10A_0016_C.avi
    OK
[ROTATE] _D (270° CW): 10A_0016_A.avi → 10A_0016_D.avi
    OK
[RENAME] 10A_0205.avi → 10A_0205_A.avi
[ROTATE] _B (90° CW): 10A_0205_A.avi → 10A_0205_B.avi
    OK
[ROTATE] _C (180°): 10A_0205_A.avi → 10A_0205_C.avi
    OK
[ROTATE] _D (270° CW): 10A_0205_A.avi → 10A_0205_D.avi
    OK
[RENAME] 10A_0456.avi → 10A_0456_A.avi
[ROTATE] _B (90° CW): 10A_0456_A.avi → 10A_0456_B.avi
    OK
[ROTATE] _C (180°): 10A_0456_A.avi → 10A_0456_C.avi
    OK
[ROTATE] _D (270° CW): 10A_0456_A.avi → 10A_0456_D.avi
    OK
[RENAME] 10B_0008.avi → 10B_0008_A.avi
[ROTATE] _B (90° CW): 10B_0008_A.avi → 10B_0008_B.avi
    OK
[ROTATE] _C (180°): 10B_0008_A.avi → 10B_0008_C.avi
    OK
[ROTATE] _D (270° CW): 10B_0008_A.avi → 10B_0008_D.avi
    OK
[

In [42]:

import os
import re
import shutil
import subprocess
from typing import Tuple  # ✅ for Python < 3.9

SUFFIX_PATTERN = re.compile(r"^(?P<base>.+)_(?P<sfx>[ABCD])\.avi$", re.IGNORECASE)
SUFFIX_MAP = {"A": "E", "B": "F", "C": "G", "D": "H"}

def is_avi(filename: str) -> bool:
    return filename.lower().endswith(".avi")

def ffmpeg_available() -> bool:
    return shutil.which("ffmpeg") is not None

def make_flipped(src: str, dst: str, codec: str = "mpeg4", qscale: int = 2) -> Tuple[bool, str]:
    """
    Run FFmpeg to horizontally flip video into an AVI container.
    - Video re-encoded as MPEG-4 (common for AVI) with qscale controlling quality.
    - Audio stream copied if present.
    """
    if not ffmpeg_available():
        return False, "ffmpeg not found in PATH."

    # Ensure destination directory exists
    os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)

    cmd = [
        "ffmpeg",
        "-hide_banner", "-loglevel", "error",  # cleaner logs
        "-y",                                  # overwrite (we only call when dst doesn't exist)
        "-i", src,
        "-vf", "hflip",
        "-c:v", codec, "-qscale:v", str(qscale),
        "-c:a", "copy",
        dst,
    ]

    try:
        subprocess.run(cmd, check=True)
        return True, "Flipped successfully."
    except subprocess.CalledProcessError as e:
        return False, f"FFmpeg error: {e}"
    except Exception as e:
        return False, f"Unexpected error: {e}"

def process_horizontal_flips(output_dir: str, dry_run: bool = False):
    """
    Flip (left-right) every .avi whose name ends with _A/_B/_C/_D.
    Write outputs with suffix _E/_F/_G/_H respectively.
    Skip any files not matching that pattern, and skip existing outputs.
    """
    files = sorted(f for f in os.listdir(output_dir) if is_avi(f))

    if not files:
        print("[INFO] No AVI files found.")
        return

    print(f"[INFO] Found {len(files)} AVI files in {output_dir}")

    for fname in files:
        m = SUFFIX_PATTERN.match(fname)
        if not m:
            # Not one of A/B/C/D → skip safely
            # e.g., 'video.avi', '1A_0152_.avi', '1A_0152_T.avi', '1A_0152_E.avi', etc.
            continue

        src_suffix = m.group("sfx").upper()
        dst_suffix = SUFFIX_MAP[src_suffix]

        src_path = os.path.join(output_dir, fname)
        dst_name = f"{m.group('base')}_{dst_suffix}.avi"
        dst_path = os.path.join(output_dir, dst_name)

        if os.path.exists(dst_path):
            print(f"[SKIP] Exists: {dst_name}")
            continue

        print(f"[FLIP] hflip: {fname} → {dst_name}")
        if dry_run:
            continue

        ok, msg = make_flipped(src_path, dst_path)


In [43]:

# Set your directory containing the AVI files
output_dir = r"/Volumes/Green SSD/00Workspace_portable/videos_aug/"
output_dir= '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug'
output_dir= '/Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj'



# Preview actions without writing files:
#process_horizontal_flips(output_dir, dry_run=True)

# Do the actual flipping:
process_horizontal_flips(output_dir, dry_run=False)


[INFO] Found 308 AVI files in /Volumes/Green SSD/00Workspace_portable/segmented_videos/bot_clips_seg/videos_aug_obj
[FLIP] hflip: 10A_0016_A.avi → 10A_0016_E.avi
[FLIP] hflip: 10A_0016_B.avi → 10A_0016_F.avi
[FLIP] hflip: 10A_0016_C.avi → 10A_0016_G.avi
[FLIP] hflip: 10A_0016_D.avi → 10A_0016_H.avi
[FLIP] hflip: 10A_0205_A.avi → 10A_0205_E.avi
[FLIP] hflip: 10A_0205_B.avi → 10A_0205_F.avi
[FLIP] hflip: 10A_0205_C.avi → 10A_0205_G.avi
[FLIP] hflip: 10A_0205_D.avi → 10A_0205_H.avi
[FLIP] hflip: 10A_0456_A.avi → 10A_0456_E.avi
[FLIP] hflip: 10A_0456_B.avi → 10A_0456_F.avi
[FLIP] hflip: 10A_0456_C.avi → 10A_0456_G.avi
[FLIP] hflip: 10A_0456_D.avi → 10A_0456_H.avi
[FLIP] hflip: 10B_0008_A.avi → 10B_0008_E.avi
[FLIP] hflip: 10B_0008_B.avi → 10B_0008_F.avi
[FLIP] hflip: 10B_0008_C.avi → 10B_0008_G.avi
[FLIP] hflip: 10B_0008_D.avi → 10B_0008_H.avi
[FLIP] hflip: 10B_0032_A.avi → 10B_0032_E.avi
[FLIP] hflip: 10B_0032_B.avi → 10B_0032_F.avi
[FLIP] hflip: 10B_0032_C.avi → 10B_0032_G.avi
[FLIP] hfl

In [34]:
# 45 degrees filitered canvas
import os
import re
import cv2
import numpy as np
from typing import Dict, Tuple, List

# -----------------------------
# Config & filename recognition
# -----------------------------

# Match only base names ending in _A.avi (case-insensitive)
A_PATTERN = re.compile(r"^(?P<base>.+)_A\.avi$", re.IGNORECASE)

# Map the desired CW angles to output suffixes
# OpenCV rotate expects degrees; CW is positive if we rotate the image accordingly,
# but we'll compute a proper rotation matrix and expand canvas.
ANGLE_TO_SUFFIX: List[Tuple[int, str]] = [
    (45,  "W"),   # 45° CW
    (135, "X"),   # 135° CW
    (225, "Y"),   # 225° CW
    (315, "Z"),   # 315° CW
]

def is_avi(filename: str) -> bool:
    return filename.lower().endswith(".avi")

# -----------------------------
# Geometry helpers
# -----------------------------

def _rotation_geometry(w: int, h: int, angle_deg: int) -> Dict[str, Tuple[int, int]]:
    """
    Compute rotation matrix and expanded bounds so that the whole rotated image fits.
    Returns dict with keys:
      - M: 2x3 rotation matrix (centered & translated for expanded canvas)
      - new_w, new_h: expanded canvas size after rotation
    """
    # Rotation center in original frame
    center = (w / 2.0, h / 2.0)

    # Standard rotation matrix around center (CCW in OpenCV).
    # For CW rotation by angle_deg, OpenCV's rotation is CCW by negative angle.
    # But "CW 45°" visually is same as "CCW -45°". We'll pass negative angle to get CW.
    M = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)  # negative for CW

    # Compute expanded canvas size
    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)

    # Adjust matrix translation to put rotated image into the center of new canvas
    M[0, 2] += (new_w / 2.0) - center[0]
    M[1, 2] += (new_h / 2.0) - center[1]

    return {"M": M, "new_w": new_w, "new_h": new_h}

def _prepare_mask(w: int, h: int, M: np.ndarray, new_w: int, new_h: int) -> np.ndarray:
    """
    Build a binary mask (0/255) for the rotated foreground region that is independent of content.
    We warp an all-ones image with the same transform to get exact coverage.
    """
    mask_src = np.full((h, w), 255, dtype=np.uint8)  # white (foreground)
    mask_rot = cv2.warpAffine(
        mask_src, M, (new_w, new_h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    return mask_rot

def _fit_dimensions(new_w: int, new_h: int, target_w: int, target_h: int) -> Tuple[int, int, float]:
    """
    Compute a scale factor to fit (new_w,new_h) fully inside (target_w,target_h) preserving aspect.
    Return (fit_w, fit_h, scale).
    """
    if new_w == 0 or new_h == 0:
        return target_w, target_h, 1.0
    scale = min(target_w / float(new_w), target_h / float(new_h))
    fit_w = max(1, int(round(new_w * scale)))
    fit_h = max(1, int(round(new_h * scale)))
    return fit_w, fit_h, scale

def _kernel_from_radius(radius: int) -> Tuple[int, int]:
    """
    Build a Gaussian blur kernel from a 'radius'-like parameter.
    Kernel must be odd and positive.
    """
    k = max(1, 2 * radius + 1)
    if k % 2 == 0:
        k += 1
    return (k, k)

# -----------------------------
# Frame processing (OpenCV)
# -----------------------------

def rotate_and_composite_frame(
    frame_bgr: np.ndarray,
    geom: Dict[str, Tuple[int, int]],
    mask_rot: np.ndarray,
    fit_w: int, fit_h: int,
    x0: int, y0: int,
    blur_radius: int
) -> np.ndarray:
    """
    Rotate frame using precomputed geometry, resize to fit original canvas,
    and composite onto a blurred background using precomputed mask.
    Returns the final composited frame (BGR).
    """
    h, w = frame_bgr.shape[:2]
    M = geom["M"]
    new_w = geom["new_w"]
    new_h = geom["new_h"]

    # 1) Build blurred background from the current frame (same size as original)
    ksize = _kernel_from_radius(blur_radius)
    bg_blur = cv2.GaussianBlur(frame_bgr, ksize, sigmaX=0, sigmaY=0)

    # 2) Rotate the foreground with expanded canvas
    rotated = cv2.warpAffine(
        frame_bgr, M, (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)  # black corners that will be masked out
    )

    # 3) Resize rotated foreground to fit the original canvas
    rot_fit = cv2.resize(rotated, (fit_w, fit_h), interpolation=cv2.INTER_LINEAR)

    # 4) Resize mask (precomputed once) to the same fitted size
    mask_fit = cv2.resize(mask_rot, (fit_w, fit_h), interpolation=cv2.INTER_NEAREST)

    # 5) Composite centered using mask
    # Ensure mask is binary 0 or 255
    _, mask_bin = cv2.threshold(mask_fit, 1, 255, cv2.THRESH_BINARY)
    mask_3c = cv2.merge([mask_bin] * 3)  # to 3 channels

    roi = bg_blur[y0:y0 + fit_h, x0:x0 + fit_w]
    # Combine using mask: where mask==255 take rot_fit, else keep roi
    composite_roi = np.where(mask_3c == 255, rot_fit, roi)
    bg_blur[y0:y0 + fit_h, x0:x0 + fit_w] = composite_roi

    return bg_blur

# -----------------------------
# Video processing (per file)
# -----------------------------

def process_single_A_to_WXYZ_opencv(
    src_path: str,
    angle_suffixes: List[Tuple[int, str]] = ANGLE_TO_SUFFIX,
    blur_radius: int = 25,
    codec_fourcc: str = "MJPG"
) -> None:
    """
    Process a single _A.avi file and write _W/_X/_Y/_Z outputs using OpenCV.
    - Keeps original width/height and fps.
    - Does NOT preserve audio (OpenCV limitation).
    - Skips outputs that already exist.
    """
    src_name = os.path.basename(src_path)
    if not src_name.lower().endswith("_a.avi"):
        print(f"[WARN] Not an _A source: {src_name}")
        return

    folder = os.path.dirname(src_path)
    base = src_name[:-6]  # strip "_A.avi"

    cap = cv2.VideoCapture(src_path)
    if not cap.isOpened():
        print(f"[FAIL] Cannot open: {src_name}")
        return

    # Get properties
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-3:
        fps = 30.0  # fallback

    print(f"[INFO] Opened {src_name} ({W}x{H} @ {fps:.2f} fps)")

    # Prepare per-angle geometry & outputs
    tasks = []
    for angle_deg, suffix in angle_suffixes:
        out_name = f"{base}_{suffix}.avi"
        out_path = os.path.join(folder, out_name)
        if os.path.exists(out_path):
            print(f"[SKIP] Exists: {out_name}")
            continue

        # Geometry for rotation
        geom = _rotation_geometry(W, H, angle_deg)

        # Precompute mask for rotated region (independent of content)
        mask_rot = _prepare_mask(W, H, geom["M"], geom["new_w"], geom["new_h"])

        # Fit rotated foreground back to original canvas, and compute center offsets
        fit_w, fit_h, _ = _fit_dimensions(geom["new_w"], geom["new_h"], W, H)
        x0 = (W - fit_w) // 2
        y0 = (H - fit_h) // 2

        # Prepare writer
        fourcc = cv2.VideoWriter_fourcc(*codec_fourcc)
        writer = cv2.VideoWriter(out_path, fourcc, fps, (W, H))
        if not writer.isOpened():
            print(f"[FAIL] Cannot create output: {out_name}")
            continue

        tasks.append({
            "suffix": suffix,
            "out_path": out_path,
            "writer": writer,
            "angle": angle_deg,
            "geom": geom,
            "mask_rot": mask_rot,
            "fit_w": fit_w,
            "fit_h": fit_h,
            "x0": x0,
            "y0": y0,
        })
        print(f"[WRITE] {src_name} → {out_name}  (CW {angle_deg}°)")

    if not tasks:
        print(f"[INFO] Nothing to do for {src_name}.")
        cap.release()
        return

    # Process frames
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        for t in tasks:
            out_frame = rotate_and_composite_frame(
                frame_bgr=frame,
                geom=t["geom"],
                mask_rot=t["mask_rot"],
                fit_w=t["fit_w"],
                fit_h=t["fit_h"],
                x0=t["x0"],
                y0=t["y0"],
                blur_radius=blur_radius
            )
            t["writer"].write(out_frame)

        frame_idx += 1
        # Optional: print progress every N frames
        if frame_idx % 300 == 0:
            print(f"[PROGRESS] {src_name}: {frame_idx} frames processed...")

    # Cleanup
    cap.release()
    for t in tasks:
        t["writer"].release()
        print(f"[DONE] {os.path.basename(t['out_path'])}")

# -----------------------------
# Batch processor for a folder
# -----------------------------

def process_rotations_from_A_to_WXYZ_opencv(
    output_dir: str,
    blur_radius: int = 25,
    codec_fourcc: str = "MJPG"
) -> None:
    """
    For each AVI ending with _A.avi in output_dir:
      - Create blurred-corner rotations (clockwise) at 45°, 135°, 225°, 315°.
      - Write outputs as _W, _X, _Y, _Z respectively.
      - Skip existing outputs and skip non-target files.
    """
    files = sorted(f for f in os.listdir(output_dir) if is_avi(f))
    if not files:
        print("[INFO] No AVI files found.")
        return

    print(f"[INFO] Found {len(files)} AVI files in {output_dir}")

    for fname in files:
        m = A_PATTERN.match(fname)
        if not m:
            continue  # only process _A sources

        src_path = os.path.join(output_dir, fname)
        process_single_A_to_WXYZ_opencv(
            src_path=src_path,
            angle_suffixes=ANGLE_TO_SUFFIX,
            blur_radius=blur_radius,
            codec_fourcc=codec_fourcc
        )


In [36]:

# 1) Test on a single file
#single_src = r"/Volumes/Green SSD/00Workspace_portable/videos_aug_test/1A_0152_A.avi"
#process_single_A_to_WXYZ_opencv(single_src, blur_radius=25, codec_fourcc="MJPG")

# 2) Run on a folder (only _A.avi files)
output_dir = r"/Volumes/Green SSD/00Workspace_portable/videos_aug/"
#output_dir = r"/Volumes/Green SSD/00Workspace_portable/videos_aug_test/"
process_rotations_from_A_to_WXYZ_opencv(output_dir, blur_radius=40, codec_fourcc="MJPG")



[INFO] Found 1572 AVI files in /Volumes/Green SSD/00Workspace_portable/videos_aug/
[INFO] Opened 10A_0016_A.avi (1280x1024 @ 7.00 fps)
[SKIP] Exists: 10A_0016_W.avi
[SKIP] Exists: 10A_0016_X.avi
[SKIP] Exists: 10A_0016_Y.avi
[SKIP] Exists: 10A_0016_Z.avi
[INFO] Nothing to do for 10A_0016_A.avi.
[INFO] Opened 10A_0205_A.avi (1280x1024 @ 7.00 fps)
[SKIP] Exists: 10A_0205_W.avi
[SKIP] Exists: 10A_0205_X.avi
[SKIP] Exists: 10A_0205_Y.avi
[SKIP] Exists: 10A_0205_Z.avi
[INFO] Nothing to do for 10A_0205_A.avi.
[INFO] Opened 10A_0456_A.avi (1280x1024 @ 7.00 fps)
[SKIP] Exists: 10A_0456_W.avi
[SKIP] Exists: 10A_0456_X.avi
[SKIP] Exists: 10A_0456_Y.avi
[SKIP] Exists: 10A_0456_Z.avi
[INFO] Nothing to do for 10A_0456_A.avi.
[INFO] Opened 10B_0008_A.avi (1280x1024 @ 7.00 fps)
[SKIP] Exists: 10B_0008_W.avi
[SKIP] Exists: 10B_0008_X.avi
[SKIP] Exists: 10B_0008_Y.avi
[SKIP] Exists: 10B_0008_Z.avi
[INFO] Nothing to do for 10B_0008_A.avi.
[INFO] Opened 10B_0032_A.avi (1280x1024 @ 7.00 fps)
[SKIP] Exists

In [13]:

# 45 degrees blank background (solid black canvas)

import os
import re
import cv2
import numpy as np
from typing import Dict, Tuple, List

# -----------------------------
# Config & filename recognition
# -----------------------------

# Match only base names ending in _A.avi (case-insensitive)
A_PATTERN = re.compile(r"^(?P<base>.+)_A\.avi$", re.IGNORECASE)

# Map the desired CW angles to output suffixes
ANGLE_TO_SUFFIX: List[Tuple[int, str]] = [
    (45,  "W"),   # 45° CW
    (135, "X"),   # 135° CW
    (225, "Y"),   # 225° CW
    (315, "Z"),   # 315° CW
]

def is_avi(filename: str) -> bool:
    return filename.lower().endswith(".avi")

# -----------------------------
# Geometry helpers
# -----------------------------

def _rotation_geometry(w: int, h: int, angle_deg: int) -> Dict[str, Tuple[int, int]]:
    """
    Compute rotation matrix and expanded bounds so that the whole rotated image fits.
    Returns dict with keys:
      - M: 2x3 rotation matrix (centered & translated for expanded canvas)
      - new_w, new_h: expanded canvas size after rotation
    """
    center = (w / 2.0, h / 2.0)

    # For CW rotation by angle_deg, pass negative angle to OpenCV (which is CCW by default).
    M = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)

    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)

    # Adjust matrix to center the rotated image in the expanded canvas
    M[0, 2] += (new_w / 2.0) - center[0]
    M[1, 2] += (new_h / 2.0) - center[1]

    return {"M": M, "new_w": new_w, "new_h": new_h}

def _prepare_mask(w: int, h: int, M: np.ndarray, new_w: int, new_h: int) -> np.ndarray:
    """
    Build a binary mask (0/255) for the rotated foreground region that is independent of content.
    """
    mask_src = np.full((h, w), 255, dtype=np.uint8)  # white (foreground)
    mask_rot = cv2.warpAffine(
        mask_src, M, (new_w, new_h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    return mask_rot

def _fit_dimensions(new_w: int, new_h: int, target_w: int, target_h: int) -> Tuple[int, int, float]:
    """
    Compute a scale factor to fit (new_w,new_h) fully inside (target_w,target_h) preserving aspect.
    Return (fit_w, fit_h, scale).
    """
    if new_w == 0 or new_h == 0:
        return target_w, target_h, 1.0
    scale = min(target_w / float(new_w), target_h / float(new_h))
    fit_w = max(1, int(round(new_w * scale)))
    fit_h = max(1, int(round(new_h * scale)))
    return fit_w, fit_h, scale

# -----------------------------
# Frame processing (OpenCV)
# -----------------------------

def rotate_and_composite_frame(
    frame_bgr: np.ndarray,
    geom: Dict[str, Tuple[int, int]],
    mask_rot: np.ndarray,
    fit_w: int, fit_h: int,
    x0: int, y0: int,
    blur_radius: int  # kept for compatibility; unused in blank background mode
) -> np.ndarray:
    """
    Rotate frame using precomputed geometry, resize to fit original canvas,
    and composite onto a **blank (black)** background using precomputed mask.
    Returns the final composited frame (BGR).
    """
    h, w = frame_bgr.shape[:2]
    M = geom["M"]
    new_w = geom["new_w"]
    new_h = geom["new_h"]

    # 1) Create a blank black background (same size as original)
    bg_black = np.zeros((h, w, 3), dtype=np.uint8)

    # 2) Rotate the foreground with expanded canvas
    rotated = cv2.warpAffine(
        frame_bgr, M, (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)  # black corners
    )

    # 3) Resize rotated foreground to fit the original canvas
    rot_fit = cv2.resize(rotated, (fit_w, fit_h), interpolation=cv2.INTER_LINEAR)

    # 4) Resize mask (precomputed once) to the same fitted size
    mask_fit = cv2.resize(mask_rot, (fit_w, fit_h), interpolation=cv2.INTER_NEAREST)

    # 5) Composite centered using mask
    _, mask_bin = cv2.threshold(mask_fit, 1, 255, cv2.THRESH_BINARY)
    mask_3c = cv2.merge([mask_bin] * 3)  # to 3 channels

    roi = bg_black[y0:y0 + fit_h, x0:x0 + fit_w]
    composite_roi = np.where(mask_3c == 255, rot_fit, roi)
    bg_black[y0:y0 + fit_h, x0:x0 + fit_w] = composite_roi

    return bg_black

# -----------------------------
# Video processing (per file)
# -----------------------------

def process_single_A_to_WXYZ_opencv(
    src_path: str,
    angle_suffixes: List[Tuple[int, str]] = ANGLE_TO_SUFFIX,
    blur_radius: int = 25,      # kept for compatibility; ignored
    codec_fourcc: str = "MJPG"
) -> None:
    """
    Process a single _A.avi file and write _W/_X/_Y/_Z outputs using OpenCV.
    - Keeps original width/height and fps.
    - Does NOT preserve audio (OpenCV limitation).
    - Skips outputs that already exist.
    - Uses **blank black background** (no blur).
    """
    src_name = os.path.basename(src_path)
    if not src_name.lower().endswith("_a.avi"):
        print(f"[WARN] Not an _A source: {src_name}")
        return

    folder = os.path.dirname(src_path)
    base = src_name[:-6]  # strip "_A.avi"

    cap = cv2.VideoCapture(src_path)
    if not cap.isOpened():
        print(f"[FAIL] Cannot open: {src_name}")
        return

    # Get properties
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-3:
        fps = 30.0  # fallback

    print(f"[INFO] Opened {src_name} ({W}x{H} @ {fps:.2f} fps)")

    # Prepare per-angle geometry & outputs
    tasks = []
    for angle_deg, suffix in angle_suffixes:
        out_name = f"{base}_{suffix}.avi"
        out_path = os.path.join(folder, out_name)
        if os.path.exists(out_path):
            print(f"[SKIP] Exists: {out_name}")
            continue

        # Geometry for rotation
        geom = _rotation_geometry(W, H, angle_deg)

        # Precompute mask for rotated region (independent of content)
        mask_rot = _prepare_mask(W, H, geom["M"], geom["new_w"], geom["new_h"])

        # Fit rotated foreground back to original canvas, and compute center offsets
        fit_w, fit_h, _ = _fit_dimensions(geom["new_w"], geom["new_h"], W, H)
        x0 = (W - fit_w) // 2
        y0 = (H - fit_h) // 2

        # Prepare writer
        fourcc = cv2.VideoWriter_fourcc(*codec_fourcc)
        writer = cv2.VideoWriter(out_path, fourcc, fps, (W, H))
        if not writer.isOpened():
            print(f"[FAIL] Cannot create output: {out_name}")
            continue

        tasks.append({
            "suffix": suffix,
            "out_path": out_path,
            "writer": writer,
            "angle": angle_deg,
            "geom": geom,
            "mask_rot": mask_rot,
            "fit_w": fit_w,
            "fit_h": fit_h,
            "x0": x0,
            "y0": y0,
        })
        print(f"[WRITE] {src_name} → {out_name}  (CW {angle_deg}°)")

    if not tasks:
        print(f"[INFO] Nothing to do for {src_name}.")
        cap.release()
        return

    # Process frames
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        for t in tasks:
            out_frame = rotate_and_composite_frame(
                frame_bgr=frame,
                geom=t["geom"],
                mask_rot=t["mask_rot"],
                fit_w=t["fit_w"],
                fit_h=t["fit_h"],
                x0=t["x0"],
                y0=t["y0"],
                blur_radius=blur_radius  # ignored
            )
            t["writer"].write(out_frame)

        frame_idx += 1
        if frame_idx % 300 == 0:
            print(f"[PROGRESS] {src_name}: {frame_idx} frames processed...")

    # Cleanup
    cap.release()
    for t in tasks:
        t["writer"].release()
        print(f"[DONE] {os.path.basename(t['out_path'])}")

# -----------------------------
# Batch processor for a folder
# -----------------------------

def process_rotations_from_A_to_WXYZ_opencv(
    output_dir: str,
    blur_radius: int = 25,  # kept for compatibility; ignored
    codec_fourcc: str = "MJPG"
) -> None:
    """
    For each AVI ending with _A.avi in output_dir:
      - Create rotated videos (clockwise) at 45°, 135°, 225°, 315°.
      - Write outputs as _W, _X, _Y, _Z respectively.
      - Skip existing outputs and skip non-target files.
      - Uses **blank black background** (no blur).
    """
    files = sorted(f for f in os.listdir(output_dir) if is_avi(f))
    if not files:
        print("[INFO] No AVI files found.")
        return

    print(f"[INFO] Found {len(files)} AVI files in {output_dir}")

    for fname in files:
        m = A_PATTERN.match(fname)
        if not m:
            continue  # only process _A sources

        src_path = os.path.join(output_dir, fname)
        process_single_A_to_WXYZ_opencv(
            src_path=src_path,
            angle_suffixes=ANGLE_TO_SUFFIX,
            blur_radius=blur_radius,   # ignored
            codec_fourcc=codec_fourcc
        )


In [14]:

# 1) Test on a single file
#single_src = r"/Volumes/Green SSD/00Workspace_portable/videos_aug_test/1A_0152_A.avi"
#process_single_A_to_WXYZ_opencv(single_src, blur_radius=25, codec_fourcc="MJPG")

# 2) Run on a folder (only _A.avi files)
output_dir = r"/Volumes/Green SSD/00Workspace_portable/videos_aug/"
#output_dir = r"/Volumes/Green SSD/00Workspace_portable/videos_aug_test/"
process_rotations_from_A_to_WXYZ_opencv(output_dir, blur_radius=40, codec_fourcc="MJPG")



[INFO] Found 1128 AVI files in /Volumes/Green SSD/00Workspace_portable/videos_aug/
[INFO] Opened 10A_0016_A.avi (1280x1024 @ 7.00 fps)
[WRITE] 10A_0016_A.avi → 10A_0016_W.avi  (CW 45°)
[WRITE] 10A_0016_A.avi → 10A_0016_X.avi  (CW 135°)
[WRITE] 10A_0016_A.avi → 10A_0016_Y.avi  (CW 225°)
[WRITE] 10A_0016_A.avi → 10A_0016_Z.avi  (CW 315°)
[DONE] 10A_0016_W.avi
[DONE] 10A_0016_X.avi
[DONE] 10A_0016_Y.avi
[DONE] 10A_0016_Z.avi
[INFO] Opened 10A_0205_A.avi (1280x1024 @ 7.00 fps)
[WRITE] 10A_0205_A.avi → 10A_0205_W.avi  (CW 45°)
[WRITE] 10A_0205_A.avi → 10A_0205_X.avi  (CW 135°)
[WRITE] 10A_0205_A.avi → 10A_0205_Y.avi  (CW 225°)
[WRITE] 10A_0205_A.avi → 10A_0205_Z.avi  (CW 315°)
[DONE] 10A_0205_W.avi
[DONE] 10A_0205_X.avi
[DONE] 10A_0205_Y.avi
[DONE] 10A_0205_Z.avi
[INFO] Opened 10A_0456_A.avi (1280x1024 @ 7.00 fps)
[WRITE] 10A_0456_A.avi → 10A_0456_W.avi  (CW 45°)
[WRITE] 10A_0456_A.avi → 10A_0456_X.avi  (CW 135°)
[WRITE] 10A_0456_A.avi → 10A_0456_Y.avi  (CW 225°)
[WRITE] 10A_0456_A.avi → 10